# ViT Defect Detection - Multi-Scenario Training & Analysis

## Overview
This notebook trains a **Vision Transformer (ViT-B-16)** model across multiple training scenarios and provides comprehensive analysis including Integrated Gradients (IG) interpretation with baseline comparison.

## Quick Start

### First Time Running
1. **Update Paths**: Modify `LOCAL_PROJECT_ROOT` or `COLAB_PROJECT_ROOT` in Cell 1
2. **Activate Environment**: `conda activate CHE1148_Defect_Detecting`
3. **Run All Cells**: Execute all cells from top to bottom (Shift+Enter)

### Running on Google Colab
1. Upload notebook to Google Drive
2. Update `COLAB_PROJECT_ROOT` path to your Colab directory
3. Mount Google Drive (will prompt automatically)
4. Run all cells from top to bottom

## Notebook Structure
```
Cell 0:  Documentation (This cell)
Cell 1:  Imports & Environment Detection
Cell 2:  Paths & Device Setup
Cell 3:  Utility Functions
Cell 4:  Training Configuration
Cell 5:  Training Utilities (Early Stopping, Metrics)
Cell 6:  Data Merge & Deduplication
Cell 7:  Split Generation
Cell 8:  Label Mapping
Cell 9:  DataLoader Utilities
Cell 10: ViT Dataset Class (TextileViT)
Cell 11: ViT Model Architecture
Cell 12: Multi-Scenario Training Loop
Cell 13: Final Test Evaluation
Cell 14: Integrated Gradients Utils
Cell 15: IG Visualization
Cell 16: IG Baseline Comparison (Trained vs Untrained)
```

## Customize Model

### Modify ViT Model
Go to **Cell 11** to modify the `create_vit_model()` function:
- Use different ViT variants (vit_l_16, vit_h_14)
- Change pretrained weight strategy
- Adjust layer freezing strategy
- Add Dropout or other regularization

### Modify Training Scenarios
Edit the `scenarios` list in **Cell 12**:
```python
scenarios = [
    {"train_factor": 0, "defect_ratio": 1, "defect_classes": [...]},
    # Add your scenarios here
]
```

## Key Parameters
- `train_factor`: Controls training set size (0=full, 2=1/4, 4=1/16, 6=1/64)
- `defect_ratio`: Ratio of defective to normal samples
- `defect_classes`: List of defect types to include

## Output Files
All outputs are saved to `data/processed/output/`:
- Model weights: `best_vit_tf{factor}_dr{ratio}.pth`
- Training history: `vit_history_tf{factor}_dr{ratio}.csv`
- IG visualizations: `ig_*.png`
- Evaluation results: `final_evaluation_results.csv`

## Troubleshooting
- **Path errors**: Update paths in Cell 1
- **Captum not available**: Run `pip install captum`
- **CUDA out of memory**: Reduce batch size in Cell 4
- **Missing data files**: Ensure .h5 and .csv files exist in `data/raw/textile/`


In [ ]:
# --- Imports ---

# Standard library imports
import copy
import hashlib
import json
import math
import os
import random
import warnings
from collections import defaultdict
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple

# Third-party imports
import h5py
import numpy as np
import pandas as pd
from PIL import Image
from sklearn.metrics import average_precision_score, confusion_matrix, f1_score
from sklearn.model_selection import train_test_split
from tqdm.auto import tqdm

# PyTorch imports
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
import torchvision.transforms as T
from torchvision.models import vit_b_16, ViT_B_16_Weights

# Visualization imports
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import seaborn as sns
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

# Optional imports with safe import
try:
    import torchinfo
except ImportError:
    torchinfo = None

try:
    from captum.attr import IntegratedGradients
    from captum.attr import visualization as viz
except ImportError:
    IntegratedGradients = None
    viz = None

try:
    from transformers import AutoImageProcessor, AutoModelForImageClassification
except ImportError:
    pass

from IPython.display import display


In [ ]:
# --- Check running mode and load dataset ---

# Define project paths for Colab and local environments
# Easy to modify for different environments
COLAB_PROJECT_ROOT = Path("/content/drive/MyDrive/CHE1148_Defect_Detecting")  # Update this for your Colab path
LOCAL_PROJECT_ROOT = Path(r"E:\Download\Pycharm\CHE1148_Defect_Detecting")  # Update this for your local path

# Detect if running in Google Colab
try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    drive = None
    IN_COLAB = False

# Set project root based on environment
if IN_COLAB:
    # Mount Google Drive if not already mounted
    if not Path('/content/drive').exists():
        drive.mount('/content/drive')
    PROJECT_ROOT = COLAB_PROJECT_ROOT
    runtime_mode = "colab_drive"
else:
    PROJECT_ROOT = LOCAL_PROJECT_ROOT
    runtime_mode = "local"

# Validate project root and set working directory
if not PROJECT_ROOT.exists():
    raise FileNotFoundError(f"Project root not found: {PROJECT_ROOT}. Please update the path in Cell 1.")

os.chdir(PROJECT_ROOT)
print(f"Runtime mode: {runtime_mode}")
print(f"Working directory: {os.getcwd()}")
print(f"Project root: {PROJECT_ROOT}")


In [ ]:
# --- Paths and Device Setup ---

# Use PROJECT_ROOT from previous cell
DATA_ROOT = PROJECT_ROOT / "data"
DATA_ROOT.mkdir(parents=True, exist_ok=True)

# Data directory structure
RAW = DATA_ROOT / "raw" / "textile"
PROCESSED = DATA_ROOT / "processed"
PROCESSED.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR = PROCESSED / "output"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# File path definitions
TRAIN_H5, TRAIN_CSV = RAW / "train64.h5", RAW / "train64.csv"
TEST_H5, TEST_CSV = RAW / "test64.h5", RAW / "test64.csv"
OUT_H5, OUT_CSV = PROCESSED / "full64.h5", PROCESSED / "full64.csv"
TRAIN_SPLIT_CSV = PROCESSED / "train_split.csv"
VAL_SPLIT_CSV = PROCESSED / "val_split.csv"
TEST_SPLIT_CSV = PROCESSED / "test_split.csv"

# Device setup
def select_device(require_cuda: bool = True):
    if torch.cuda.is_available() and torch.cuda.device_count() > 0:
        return torch.device("cuda"), f"cuda ({torch.cuda.get_device_name(0)})"
    if require_cuda: raise RuntimeError("CUDA unavailable")
    return torch.device("cpu"), "cpu"

# Set to True to use GPU, False for CPU only
REQUIRE_CUDA = False
device, device_name = select_device(require_cuda=REQUIRE_CUDA)
print(f"Using device: {device_name}")

In [ ]:
# --- Utility Functions ---

# Check if a file exists, raise error if missing
def _require_file(path: Path) -> None:
    if not path.exists():
        raise FileNotFoundError(f"Missing required file: {path}")

# Convert label to string and remove leading/trailing whitespace
def _normalize_label(x) -> str:
    return str(x).strip()

# Print the number of samples per class in a DataFrame
def print_class_counts(df: pd.DataFrame, title: str) -> None:
    if "indication_type" not in df.columns:
        return
    vc = df["indication_type"].astype(str).str.strip().value_counts()
    print(f"\n[{title}] total_images={len(df)}")
    for k, v in vc.items():
        print(f"  {k}: {v}")


In [ ]:
# --- Global Training and Evaluation Config ---

# Reproducibility
GLOBAL_SEED = 42

def set_seed(seed: int = GLOBAL_SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    if hasattr(torch.backends, "cudnn"):
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = True

set_seed(GLOBAL_SEED)

FULL_CLASSES = ["good", "color", "cut", "hole", "thread", "metal_contamination"]

def _get_default_workers():
    return 2 if bool(globals().get("IN_COLAB", False)) else 0

DEFAULT_NUM_WORKERS = property(lambda self: _get_default_workers()) if False else _get_default_workers()

# --- Global Optimizer Configuration ---
OPTIM_CFG = {
    "name": "adam",
    "lr": float(0.001),
    "foreach": bool(False),
}

# --- Global Evaluation Configuration ---
EVAL_CFG = {
    "f1_average": "macro",
    "auprc_average": "macro",
    "zero_division": int(0),
    "early_stop_metric": "f1",
    "early_stop_mode": "max",
}

# --- Global Training Configuration ---
TRAIN_CFG = {
    "batch": int(1024),
    "epochs": int(20),
    "patience": int(3),
    "num_workers": 0,
    "use_fp16": True,
}

In [ ]:
# --- Training Utilities ---

# Monitors a metric and stops training if it stops improving
class EarlyStopping:
    def __init__(self, patience=5, verbose=True, mode="min", metric_name="metric", min_delta=0.0):
        if mode not in {"min", "max"}: raise ValueError("mode must be 'min' or 'max'")
        self.patience, self.verbose, self.mode = patience, verbose, mode
        self.metric_name, self.min_delta = metric_name, min_delta
        self.counter, self.early_stop = 0, False
        self.best_score = float("inf") if mode == "min" else -float("inf")
        self.best_model_state = None

    # Check if the current score improves upon the best recorded score
    def _is_improvement(self, score):
        if self.mode == "min": return score < (self.best_score - self.min_delta)
        return score > (self.best_score + self.min_delta)

    # Update early stopping status based on the latest validation score
    def __call__(self, score, model):
        if self._is_improvement(score):
            self.best_score, self.counter = score, 0
            self.best_model_state = copy.deepcopy(model.state_dict())
            if self.verbose: print(f"Validation {self.metric_name} improved. Saving weights.")
        else:
            self.counter += 1
            if self.verbose: print(f"EarlyStopping counter: {self.counter} of {self.patience}")
            if self.counter >= self.patience: self.early_stop = True

# Compute macro AUPRC only for classes present in y_true
def _compute_macro_auprc_safe(y_true, y_prob, num_classes):
    y_true_bin = np.eye(num_classes)[y_true]
    ap_values = []

    for cls_id in range(num_classes):
        y_cls = y_true_bin[:, cls_id]
        if not np.any(y_cls == 1):
            continue
        try:
            with warnings.catch_warnings():  # type: ignore[var-annotated]
                warnings.filterwarnings(
                    "ignore",
                    message="No positive class found in y_true, recall is set to one for all thresholds.",
                    category=UserWarning,
                )
                ap = average_precision_score(y_cls, y_prob[:, cls_id])
            ap_values.append(float(ap))
        except Exception:
            continue

    if not ap_values:
        return float("nan")
    return float(np.mean(ap_values))

# Calculate Accuracy, F1-score, and AUPRC
def _compute_eval_metrics(y_true, y_pred, y_prob, num_classes):
    acc = 100.0 * (y_pred == y_true).sum() / max(len(y_true), 1)
    f1 = f1_score(y_true, y_pred, labels=list(range(num_classes)),
                  average=EVAL_CFG["f1_average"], zero_division=int(EVAL_CFG["zero_division"]))
    auprc = _compute_macro_auprc_safe(y_true, y_prob, num_classes)
    return {"accuracy": float(acc), "f1": float(f1), "auprc": float(auprc)}

# Convert a numeric metric value to a formatted string, handling NaNs
def _metric_to_str(v, digits=4):
    return "nan" if np.isnan(v) else f"{v:.{digits}f}"

# Execute a single training or validation epoch across the provided loader
def run_step(model, loader, criterion, optimizer, device, is_train=True):
    model.train() if is_train else model.eval()
    total_loss, all_true, all_pred, all_prob = 0.0, [], [], []

    with torch.set_grad_enabled(is_train):
        for imgs, labels in loader:
            imgs, labels = imgs.to(device), labels.to(device)
            outputs = model(imgs)
            loss = criterion(outputs, labels)

            if is_train:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

            total_loss += loss.item()
            all_true.append(labels.cpu().numpy())
            all_pred.append(outputs.argmax(dim=1).cpu().numpy())
            all_prob.append(torch.softmax(outputs, dim=1).detach().cpu().numpy())

    avg_loss = total_loss / max(len(loader), 1)
    if not all_true: return avg_loss, {"accuracy": 0.0, "f1": 0.0, "auprc": float("nan")}

    y_true, y_pred, y_prob = np.concatenate(all_true), np.concatenate(all_pred), np.concatenate(all_prob)
    return avg_loss, _compute_eval_metrics(y_true, y_pred, y_prob, y_prob.shape[1])

In [ ]:
# --- Merge and Deduplication ---

def merge_data() -> None:
    if OUT_H5.exists() and OUT_CSV.exists():
        print("Dataset already merged.")
        return

    _require_file(TRAIN_CSV); _require_file(TEST_CSV)
    _require_file(TRAIN_H5); _require_file(TEST_H5)

    df_train = pd.read_csv(TRAIN_CSV)
    df_test = pd.read_csv(TEST_CSV)
    df_train["original_split"], df_test["original_split"] = "train", "test"

    full_df = pd.concat([df_train, df_test], ignore_index=True)
    full_df.to_csv(OUT_CSV, index=False)

    with h5py.File(str(OUT_H5), "w") as f_out:
        with h5py.File(str(TRAIN_H5), "r") as f_tr, h5py.File(str(TEST_H5), "r") as f_te:
            tr_imgs, te_imgs = f_tr["images"], f_te["images"]
            dset = f_out.create_dataset("images", shape=(len(tr_imgs)+len(te_imgs), *tr_imgs.shape[1:]), dtype="f")
            dset[:len(tr_imgs)] = tr_imgs[:]
            dset[len(tr_imgs):] = te_imgs[:]
    print(f"Merged data saved to: {OUT_H5}")


def get_h5_hashes(h5_path: Path, total_images: int, chunk_size: int = 5000) -> List[str]:
    hashes = [""] * total_images
    with h5py.File(str(h5_path), "r") as f:
        imgs = f["images"]
        for start in range(0, total_images, chunk_size):
            end = min(start + chunk_size, total_images)
            chunk = imgs[start:end]
            for i, img in enumerate(chunk):
                hashes[start + i] = hashlib.md5(img.tobytes()).hexdigest()
    return hashes

def analyze_duplicates() -> List[str]:
    df = pd.read_csv(OUT_CSV)
    with h5py.File(str(OUT_H5), "r") as f:
        total = int(f["images"].shape[0])

    all_hashes = get_h5_hashes(OUT_H5, total)
    hash_map = defaultdict(list)
    for idx, h in enumerate(all_hashes): hash_map[h].append(idx)

    dups = {h: idxs for h, idxs in hash_map.items() if len(idxs) > 1}
    if dups:
        dup_indices = [i for idxs in dups.values() for i in idxs]
        report_df = df.iloc[dup_indices].copy()
        report_df["md5"] = [all_hashes[i] for i in dup_indices]
        report_df.to_csv(PROCESSED / "duplicates_report.csv", index=False)

        # Check for MD5 appearing in both train and test
        leakage = report_df.groupby("md5")["original_split"].nunique()
        print("[WARNING] Leakage detected!" if (leakage > 1).any() else "[SAFE] No split leakage.")
    return all_hashes

In [ ]:
# --- Split Generation (Dedup per original split + Stratified Train/Val) ---


def reduce_training_set(dedup_df, included_classes: List[str], training_fact: int, defect_rat: float):
    """
    Reduce the training set to specified proportions of good and defective samples.
    
    Args:
        dedup_df: Deduplicated training DataFrame
        included_classes: List of class names to include
        training_fact: Training factor exponent (num_good = 8000 / 2^training_fact)
        defect_rat: Ratio of defective samples relative to good samples
    
    Returns:
        Reduced DataFrame with sampled data
    """
    # abort method if defect fraction is 0
    # Return original dataframe when no reduction is needed
    if training_fact == 0 and defect_rat == 1.0:
        return dedup_df

    # calculate number of good and defective samples
    num_good_samples = math.floor(8000 / (2 ** training_fact))
    num_defect_samples = math.floor(num_good_samples * defect_rat)

    reduced_df = pd.DataFrame(columns=dedup_df.columns)

    # sample defects and concat to df
    for i in range(1, len(included_classes)):
        holder_df = dedup_df.loc[dedup_df["indication_type"] == included_classes[i]]
        if len(holder_df) > 0:
            sample_size = min(num_defect_samples, len(holder_df))
            sampled_df = holder_df.sample(n=sample_size, random_state=42)
            reduced_df = pd.concat([reduced_df, sampled_df])

    # sample good and concat to df
    holder_df = dedup_df.loc[dedup_df["indication_type"] == included_classes[0]]
    if len(holder_df) > 0:
        sample_size = min(num_good_samples, len(holder_df))
        sampled_df = holder_df.sample(n=sample_size, random_state=42)
        reduced_df = pd.concat([reduced_df, sampled_df])

    return reduced_df


def create_clean_split(all_hashes: List[str], included_classes: List[str] = None, 
                      train_factor: int = 0, defect_ratio: float = 1.0) -> None:
    """
    Remove internal duplicates within each original split and generate Train/Val/Test CSVs.
    Remove requested classes from training set before splitting into final train and validation sets.

    Args:
        all_hashes: List of MD5 hashes for all images
        included_classes: List of classes to include (default: FULL_CLASSES)
        train_factor: Training factor exponent (0 = full, 1 = half, 2 = quarter, etc.)
        defect_ratio: Ratio of defective samples relative to good samples
    
    Outputs:
      - data/processed/train_split.csv
      - data/processed/val_split.csv
      - data/processed/test_split.csv
    """
    if included_classes is None:
        included_classes = FULL_CLASSES
        
    df = pd.read_csv(OUT_CSV).copy()
    df["abs_ptr"] = range(len(df))  # pointer into full64.h5
    df["md5"] = all_hashes
    df["indication_type"] = df["indication_type"].astype(str).str.strip()

    tr_df_raw = df[df["original_split"] == "train"].copy()
    te_df_raw = df[df["original_split"] == "test"].copy()

    # Deduplicate within each portion
    tr_before, te_before = len(tr_df_raw), len(te_df_raw)
    tr_df = tr_df_raw.drop_duplicates(subset="md5", keep="first")
    te_df = te_df_raw.drop_duplicates(subset="md5", keep="first")
    tr_removed, te_removed = tr_before - len(tr_df), te_before - len(te_df)
    total_removed = tr_removed + te_removed

    print(f"Duplicates removed (within split): train={tr_removed}, test={te_removed}, total={total_removed}")

    # Remove requested classes from intermediate training dataframe
    tr_df = tr_df[tr_df["indication_type"].isin(included_classes)]

    # Sample desired fraction of defects
    reduced_tr_df = reduce_training_set(tr_df, included_classes, train_factor, defect_ratio)

    # Stratified split (Train -> Train/Val) based on unique image index
    unique_df = reduced_tr_df.drop_duplicates("index")[["index", "indication_type"]].copy()
    
    # Check if stratification is possible
    label_counts = unique_df["indication_type"].value_counts()
    can_stratify = (label_counts.min() >= 2) and (len(label_counts) > 1)
    
    if can_stratify:
        train_idx, val_idx = train_test_split(
            unique_df["index"],
            test_size=0.1,
            random_state=42,
            stratify=unique_df["indication_type"],
        )
    else:
        print("[WARN] Stratified split disabled (insufficient per-class samples).")
        train_idx, val_idx = train_test_split(
            unique_df["index"],
            test_size=0.1,
            random_state=42,
        )

    df_train = tr_df[tr_df["index"].isin(train_idx)].sample(frac=1, random_state=42)
    df_val = tr_df[tr_df["index"].isin(val_idx)].sample(frac=1, random_state=42)

    train_path = PROCESSED / "train_split.csv"
    val_path = PROCESSED / "val_split.csv"
    test_path = PROCESSED / "test_split.csv"

    df_train.to_csv(train_path, index=False)
    df_val.to_csv(val_path, index=False)
    te_df.to_csv(test_path, index=False)

    print(f"Datasets finalized: Train({len(df_train)}), Val({len(df_val)}), Test({len(te_df)})")

    # Requested reporting
    print_class_counts(df_train, "TRAIN SPLIT")
    print_class_counts(df_val, "VAL SPLIT")


In [ ]:
# --- Label Mapping Utilities ---

# Path to save label map as JSON file
LABEL_MAP_JSON = PROCESSED / "label_map.json"

# Cache for loaded label maps to avoid rebuilding
_LABEL_MAP_CACHE: Dict[str, Dict[str, int]] = {}

# Raise error if CSV contains labels not in the label map
def _validate_labels(observed: List[str], label_map: Dict[str, int]) -> None:
    unknown = sorted(set(observed) - set(label_map.keys()))
    if unknown:
        raise ValueError(f"Unknown labels in CSV: {unknown}")

# Build label-to-index mapping from CSV, ordered by FULL_CLASSES
def build_label_map_from_full_csv(full_csv_path: Path) -> Dict[str, int]:
    df = pd.read_csv(full_csv_path)
    observed = set(df["indication_type"].astype(str).str.strip().unique().tolist())

    unknown = sorted(observed - set(FULL_CLASSES))
    if unknown:
        raise ValueError(f"CSV has labels outside FULL_CLASSES: {unknown}")

    ordered_present = [name for name in FULL_CLASSES if name in observed]
    if not ordered_present:
        raise ValueError("No valid labels found in CSV.")

    return {name: i for i, name in enumerate(ordered_present)}

# Load label map from cache, or build from CSV and optionally save to JSON
def load_or_create_label_map(csv_path: Path, persist_json: bool = True) -> Dict[str, int]:
    cache_key = str(csv_path.resolve())
    if cache_key in _LABEL_MAP_CACHE:
        return dict(_LABEL_MAP_CACHE[cache_key])

    label_map = build_label_map_from_full_csv(csv_path)
    _LABEL_MAP_CACHE[cache_key] = dict(label_map)

    if persist_json:
        LABEL_MAP_JSON.write_text(
            json.dumps(label_map, ensure_ascii=False, indent=2), encoding="utf-8"
        )
    return dict(label_map)

# Check that all labels in a split CSV exist in the label map
def validate_split_labels(csv_path: Path, label_map: Dict[str, int]) -> None:
    df = pd.read_csv(csv_path)
    observed = df["indication_type"].astype(str).str.strip().unique().tolist()
    _validate_labels([_normalize_label(x) for x in observed], label_map)

#  Validate labels in train, val, and optionally test splits
def validate_common_splits(label_map: Dict[str, int], include_test: bool = True) -> None:
    validate_split_labels(TRAIN_SPLIT_CSV, label_map)
    validate_split_labels(VAL_SPLIT_CSV, label_map)
    if include_test:
        validate_split_labels(TEST_SPLIT_CSV, label_map)


In [ ]:
# --- DataLoader Utilities ---

_TEST_LOADER_CACHE: Dict[str, Tuple["TextileViT", DataLoader, Path]] = {}

# Enable pin_memory for faster data transfer when using CUDA
def _pin_memory_enabled() -> bool:
    return isinstance(device, torch.device) and device.type == "cuda"

# Set random seeds for DataLoader workers for reproducibility
def _seed_worker(worker_id: int) -> None:
    worker_seed = (int(globals().get("GLOBAL_SEED", 42)) + worker_id) % (2**32)
    random.seed(worker_seed)
    np.random.seed(worker_seed)
    torch.manual_seed(worker_seed)

# Create a seeded PyTorch Generator for DataLoader
def _make_generator(seed_offset: int = 0) -> torch.Generator:
    gen = torch.Generator()
    gen.manual_seed(int(globals().get("GLOBAL_SEED", 42)) + seed_offset)
    return gen

# Create a DataLoader with consistent configuration for reproducibility
def _make_loader(dataset: Dataset, batch_size: int, shuffle: bool = False, seed_offset: int = 0) -> DataLoader:
    """Create a DataLoader with consistent configuration."""
    return DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        num_workers=int(TRAIN_CFG.get("num_workers", 0)),
        pin_memory=_pin_memory_enabled(),
        worker_init_fn=_seed_worker,
        generator=_make_generator(seed_offset),
    )

# Create a unique string signature for a label map (used for cache keys)
def _label_map_signature(label_map: Dict[str, int]) -> str:
    """Create a unique signature for label map caching."""
    ordered = sorted(label_map.items(), key=lambda kv: kv[1])
    return "|".join([f"{k}:{v}" for k, v in ordered])

# Create train and val datasets with their corresponding DataLoaders
def make_split_datasets_and_loaders(label_map: Dict[str, int], batch_size: int) -> Tuple["TextileViT", "TextileViT", DataLoader, DataLoader]:
    """Create ViT datasets and loaders for training."""
    train_ds = TextileViT(TRAIN_SPLIT_CSV, OUT_H5, label_map=label_map)
    val_ds = TextileViT(VAL_SPLIT_CSV, OUT_H5, label_map=label_map)
    train_loader = _make_loader(train_ds, batch_size, shuffle=True, seed_offset=0)
    val_loader = _make_loader(val_ds, batch_size, shuffle=False, seed_offset=1)
    return train_ds, val_ds, train_loader, val_loader


# Build a test loader filtered to classes present in the given label_map
def make_filtered_test_dataset_and_loader(
    label_map: Dict[str, int],
    batch_size: int,
    split_tag: str,
) -> Tuple["TextileViT", DataLoader, Path]:
    """Create filtered test dataset and loader."""
    cache_key = f"{split_tag}::{batch_size}::{_label_map_signature(label_map)}"
    if cache_key in _TEST_LOADER_CACHE:
        cached = _TEST_LOADER_CACHE[cache_key]
        return cached[0], cached[1], cached[2]

    df_test = pd.read_csv(TEST_SPLIT_CSV).copy()
    df_test["indication_type"] = df_test["indication_type"].astype(str).str.strip()
    allowed = set(label_map.keys())
    filtered = df_test[df_test["indication_type"].isin(allowed)].copy()

    filtered_csv = PROCESSED / f"test_split_filtered_{split_tag}.csv"
    filtered.to_csv(filtered_csv, index=False)

    test_ds = TextileViT(filtered_csv, OUT_H5, label_map=label_map)
    test_loader = _make_loader(test_ds, batch_size, shuffle=False, seed_offset=2)
    _TEST_LOADER_CACHE[cache_key] = (test_ds, test_loader, filtered_csv)
    return _TEST_LOADER_CACHE[cache_key][0], _TEST_LOADER_CACHE[cache_key][1], _TEST_LOADER_CACHE[cache_key][2]


# ViT-specific dataset and loader factory
def make_vit_datasets_and_loaders(label_map: Dict[str, int], batch_size: int):
    """Create ViT-specific datasets and loaders with 224x224 RGB images."""
    train_ds = TextileViT(TRAIN_SPLIT_CSV, OUT_H5, label_map=label_map)
    val_ds = TextileViT(VAL_SPLIT_CSV, OUT_H5, label_map=label_map)
    train_loader = _make_loader(train_ds, batch_size, shuffle=True, seed_offset=0)
    val_loader = _make_loader(val_ds, batch_size, shuffle=False, seed_offset=1)
    return train_ds, val_ds, train_loader, val_loader


In [ ]:
# --- ViT Dataset Definition ---


# ViT-specific dataset: converts grayscale to RGB (1→3 channels), resizes to 224x224, and normalizes
class TextileViT(Dataset):
    """Dataset class for ViT model. Converts 64x64 grayscale images to 224x224 RGB."""
    
    def __init__(self, csv_path, h5_path, label_map):
        self.df = pd.read_csv(csv_path)
        self.h5_path = str(h5_path)
        self.label_map = label_map
        self.archive = None
        # ViT transforms: resize to 224x224, convert to tensor, normalize
        self.transform = T.Compose([
            T.Resize((224, 224)),
            T.ToTensor(),
            T.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
        ])

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        if self.archive is None:
            self.archive = h5py.File(self.h5_path, "r", swmr=True)

        row = self.df.iloc[idx]
        img_np = self.archive["images"][int(row["abs_ptr"])]

        # Ensure values are 0-255
        if img_np.max() <= 1.0:
            img_np = (img_np * 255).astype("uint8")
        else:
            img_np = img_np.astype("uint8")

        # Remove extra dimensions (e.g., (64, 64, 1) -> (64, 64))
        img_np = img_np.squeeze()

        # Convert grayscale NumPy (64x64) to PIL RGB
        img = Image.fromarray(img_np).convert("RGB")

        # Apply ViT transforms: resize to 224x224 and normalize
        img = self.transform(img)

        label = self.label_map[_normalize_label(row["indication_type"])]
        return img, torch.tensor(label, dtype=torch.long)

    def __del__(self):
        """Clean up H5 file handle"""
        if self.archive is not None:
            self.archive.close()


In [ ]:
# --- Model Architecture Factory ---
# This cell contains factory functions for creating different models
# Easy to extend with new model architectures
from torchvision.models import vit_b_16, ViT_B_16_Weights


def create_vit_model(num_classes):
    """
    Create a ViT-B-16 model adapted for 3-channel 224x224 images.
    
    Args:
        num_classes: Number of output classes
    
    Returns:
        ViT model (backbone frozen, only head trainable)
    """
    weights = ViT_B_16_Weights.IMAGENET1K_V1
    model = vit_b_16(weights=weights)
    
    # Replace the classification head for our number of classes
    model.heads.head = nn.Linear(model.heads.head.in_features, num_classes)
    
    # Freeze the backbone (all parameters except the head)
    for name, param in model.named_parameters():
        if "heads.head" not in name:
            param.requires_grad = False
    
    return model


# ============================================================================
# MODEL FACTORY - Add new models here for easy experimentation
# ============================================================================

def create_resnet_model(num_classes, pretrained=True):
    """Create a ResNet-18 model for textile defect detection."""
    from torchvision.models import resnet18, ResNet18_Weights
    
    if pretrained:
        weights = ResNet18_Weights.IMAGENET1K_V1
    else:
        weights = None
    
    model = resnet18(weights=weights)
    model.conv1 = nn.Conv2d(1, 64, kernel_size=7, stride=2, padding=3, bias=False)
    model.fc = nn.Linear(model.fc.in_features, num_classes)
    return model


def create_efficientnet_model(num_classes, pretrained=True):
    """Create an EfficientNet-B2 model for textile defect detection."""
    from torchvision.models import efficientnet_b2, EfficientNet_B2_Weights
    
    if pretrained:
        weights = EfficientNet_B2_Weights.IMAGENET1K_V1
    else:
        weights = None
    
    model = efficientnet_b2(weights=weights)
    model.features[0][0] = nn.Conv2d(1, 32, kernel_size=3, stride=2, padding=1, bias=False)
    model.classifier[1] = nn.Linear(model.classifier[1].in_features, num_classes)
    return model


# Model registry - add new models here
MODEL_REGISTRY = {
    "vit": create_vit_model,
    "resnet": create_resnet_model,
    "efficientnet": create_efficientnet_model,
}


def create_model(model_name, num_classes, **kwargs):
    """
    Factory function to create models by name.
    
    Args:
        model_name: Name of the model (must be in MODEL_REGISTRY)
        num_classes: Number of output classes
        **kwargs: Additional arguments passed to the model creator
    
    Returns:
        Model instance
    """
    if model_name not in MODEL_REGISTRY:
        raise ValueError(f"Unknown model: {model_name}. Available: {list(MODEL_REGISTRY.keys())}")
    
    return MODEL_REGISTRY[model_name](num_classes, **kwargs)


In [ ]:
# --- Common Data Setup: Multi-Scenario Training ---

# Merge raw data and remove duplicates (only once)
merge_data()
hashes = analyze_duplicates()

# Define training scenarios with different split configurations
scenarios = [
    {"train_factor": 6, "defect_ratio": 1, "defect_classes": ["good","cut","color","metal_contamination","hole","thread"]}

]

# Store results for each scenario
scenario_results = {}

# Train ViT model for each scenario
for scenario_idx, scenario_cfg in enumerate(scenarios, 1):
    train_factor = scenario_cfg["train_factor"]
    defect_ratio = scenario_cfg["defect_ratio"]
    included_classes = scenario_cfg["defect_classes"]
    
    # Generate scenario name for file naming
    scenario_name = f"tf{train_factor}_dr{defect_ratio}"
    
    print(f"\n{'='*80}")
    print(f"SCENARIO {scenario_idx}/{len(scenarios)}: {scenario_name}")
    print(f"  Classes: {included_classes}")
    print(f"  Train factor: {train_factor} | Defect ratio: {defect_ratio}")
    print(f"{'='*80}\n")
    
    # Create clean split with scenario-specific parameters
    create_clean_split(hashes, included_classes, train_factor, defect_ratio)
    
    # Build and validate the label mapping for this scenario
    label_map = load_or_create_label_map(OUT_CSV)
    validate_common_splits(label_map, include_test=True)
    print(f"\nlabel_map: {label_map}")
    
    # Create ViT-specific datasets and loaders
    vit_train_ds, vit_val_ds, vit_train_loader, vit_val_loader = make_vit_datasets_and_loaders(label_map, 32)
    
    print(f"\nTraining set size: {len(vit_train_ds)}")
    print(f"Validation set size: {len(vit_val_ds)}")
    
    # Model Setup - Create actual ViT model
    model = create_vit_model(num_classes=len(label_map)).to(device)
    
    # Robust torchinfo handling
    if torchinfo:
        try:
            dummy_input = torch.randn(1, 3, 224, 224).to(device)
            print(torchinfo.summary(model, input_data=dummy_input, col_names=["input_size", "output_size", "num_params", "kernel_size"]))
        except Exception as e:
            print(f"[WARNING] torchinfo.summary failed: {e}. Proceeding to training.")
    else:
        print("[INFO] torchinfo not installed. Skipping model summary.")
    
    # Training Objects
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=float(OPTIM_CFG["lr"]), foreach=bool(OPTIM_CFG["foreach"]))
    early_stop = EarlyStopping(patience=int(TRAIN_CFG["patience"]), verbose=False, mode="max", metric_name="f1")
    
    # Training Loop
    print(f"\nStarting ViT training on: {device}")
    pbar = tqdm(range(int(TRAIN_CFG["epochs"])), desc=f"ViT [{scenario_name}]")
    history = []
    
    for epoch in pbar:
        train_loss, train_metrics = run_step(model, vit_train_loader, criterion, optimizer, device, is_train=True)
        v_loss, v_metrics = run_step(model, vit_val_loader, criterion, optimizer, device, is_train=False)
    
        v_metrics["epoch"] = epoch + 1
        history.append(v_metrics)
        pbar.set_postfix(v_f1=_metric_to_str(v_metrics["f1"]), v_acc=f"{v_metrics['accuracy']:.2f}", v_loss=f"{v_loss:.4f}")
    
        early_stop(v_metrics["f1"], model)
        if early_stop.early_stop:
            pbar.write(f"Early stopping at epoch {epoch + 1}")
            model.load_state_dict(early_stop.best_model_state)
            break
    
    # Save and Report
    best = max(history, key=lambda x: x["f1"])
    ckpt_path = OUTPUT_DIR / f"best_vit_{scenario_name}.pth"
    hist_path = OUTPUT_DIR / f"vit_history_{scenario_name}.csv"
    
    torch.save(model.state_dict(), str(ckpt_path))
    pd.DataFrame(history).to_csv(str(hist_path), index=False)
    
    scenario_results[scenario_name] = {
        "best_metrics": best,
        "checkpoint": str(ckpt_path),
        "history": hist_path,
        "config": scenario_cfg,
    }
    
    print(f"\n{'='*40}")
    print(f"SCENARIO '{scenario_name}' TRAINING COMPLETE")
    print(f"Best validation F1: {_metric_to_str(best['f1'])}")
    print(f"Best validation Acc: {best['accuracy']:.2f}%")
    print(f"Best validation AUPRC: {_metric_to_str(best['auprc'])}")
    print(f"Model saved to: {ckpt_path}")
    print(f"{'='*40}\n")

# Summary of all scenarios
print(f"\n{'='*80}")
print("ALL SCENARIOS COMPLETE - SUMMARY")
print(f"{'='*80}")
print(f"{'Scenario':<20} {'Train Factor':<15} {'Defect Ratio':<15} {'Best F1':<12} {'Best Acc':<12} {'Best AUPRC':<12}")
print(f"{'-'*80}")
for name, results in scenario_results.items():
    cfg = results["config"]
    metrics = results["best_metrics"]
    print(f"{name:<20} {cfg['train_factor']:<15} {cfg['defect_ratio']:<15} {_metric_to_str(metrics['f1']):<12} {metrics['accuracy']:<12.2f} {metrics['auprc']:<12.4f}")
print(f"{'='*80}\n")


In [ ]:
# --- Final Test Evaluation ---

# 1. Prepare Test Data
# Use the label map from the last scenario (should be consistent across scenarios)
try:
    _, test_loader, _ = make_filtered_test_dataset_and_loader(label_map, 32, split_tag="vit_final_eval")
except Exception as e:
    print(f"[ERROR] Could not create test loader: {e}")
    test_loader = None

if test_loader is not None:
    # 2. Define ViT Model Checkpoints from Scenarios
    # Map scenario names to their checkpoint and history files
    vit_models = [
        {
            "name": f"ViT (tf{cfg['train_factor']}_dr{cfg['defect_ratio']})",
            "ckpt_file": f"best_vit_tf{cfg['train_factor']}_dr{cfg['defect_ratio']}.pth",
            "hist_file": f"vit_history_tf{cfg['train_factor']}_dr{cfg['defect_ratio']}.csv",
            "config": cfg
        }
        for cfg in [scenario_results[name]["config"] for name in scenario_results]
    ]

    results = []
    criterion = nn.CrossEntropyLoss()

    print(f"\n{'='*80}")
    print(f"Evaluating ViT Models on Test Set")
    print(f"{'='*80}")

    # 3. Loop through ViT models and evaluate
    for model_cfg in vit_models:
        model_name = model_cfg["name"]
        ckpt_path = OUTPUT_DIR / model_cfg["ckpt_file"]
        hist_path = OUTPUT_DIR / model_cfg["hist_file"]

        if not ckpt_path.exists():
            print(f"[SKIP] {model_name}: Checkpoint not found at {ckpt_path}")
            continue

        print(f"\nLoading {model_name}...")
        # Initialize ViT model and load weights
        model = create_vit_model(num_classes=len(label_map)).to(device)
        model.load_state_dict(torch.load(ckpt_path, map_location=device, weights_only=True))
        model.eval()

        # Run evaluation
        test_loss, metrics = run_step(model, test_loader, criterion, None, device, is_train=False)

        # Find Best Epoch from history file
        best_epoch = "N/A"
        if hist_path.exists():
            try:
                df_hist = pd.read_csv(hist_path)
                if 'f1' in df_hist.columns and 'epoch' in df_hist.columns:
                    best_idx = df_hist['f1'].idxmax()
                    best_epoch = int(df_hist.loc[best_idx, 'epoch'])
            except Exception:
                best_epoch = "Error reading history"

        # Collect results
        results.append({
            "Model": model_name,
            "Train Factor": model_cfg["config"]["train_factor"],
            "Defect Ratio": model_cfg["config"]["defect_ratio"],
            "Best Epoch": best_epoch,
            "Test Loss": f"{test_loss:.4f}",
            "Test Accuracy": f"{metrics['accuracy']:.2f}%",
            "Test F1": f"{metrics['f1']:.4f}",
            "Test AUPRC": f"{metrics['auprc']:.4f}"
        })

    # 4. Generate Summary Table and Save
    if results:
        df_results = pd.DataFrame(results)

        # Display in console
        print(f"\n{'='*80}")
        print("VIT FINAL EVALUATION RESULTS SUMMARY")
        print(f"{'='*80}")
        print(df_results.to_string(index=False))
        print(f"{'='*80}")

        # Save to CSV
        results_csv_path = OUTPUT_DIR / "vit_final_evaluation_results.csv"
        df_results.to_csv(results_csv_path, index=False)
        print(f"\nResults saved to: {results_csv_path}")
    else:
        print("\nNo ViT models were evaluated.")
else:
    print("\nEvaluation failed because test data could not be loaded.")


In [ ]:
# Plot confusion matrix
import matplotlib.pyplot as plt
import numpy as np
from sklearn.metrics import confusion_matrix
import math

# Check if training scenarios exist
if 'scenario_results' not in globals() or not scenario_results:
    print("[ERROR] No training scenarios found. Run the training cell first.")
else:
    # Automatically detect all scenarios
    scenario_names = list(scenario_results.keys())
    num_scenarios = len(scenario_names)

    print(f"Found {num_scenarios} training scenario(s): {scenario_names}")

    # Calculate grid layout for subplots
    ncols = min(3, num_scenarios)
    nrows = math.ceil(num_scenarios / ncols)

    fig, axes = plt.subplots(nrows, ncols, figsize=(6 * ncols, 5 * nrows))
    if num_scenarios == 1:
        axes = np.array([axes])
    axes = axes.flatten()

    # Evaluate each scenario on test set
    for idx, scenario_name in enumerate(scenario_names):
        ax = axes[idx]
        scenario_cfg = scenario_results[scenario_name]["config"]
        defect_classes = scenario_cfg.get("defect_classes", FULL_CLASSES)

        # Build label map for this scenario
        scenario_label_map = {name: i for i, name in enumerate(defect_classes)}
        num_classes = len(defect_classes)

        # Create test dataset and loader
        test_ds = TextileViT(TEST_SPLIT_CSV, OUT_H5, label_map=scenario_label_map)
        test_loader = _make_loader(test_ds, batch_size=32, shuffle=False, seed_offset=2)

        # Load model for this scenario
        model = create_vit_model(num_classes=num_classes).to(device)
        ckpt_path = OUTPUT_DIR / f"best_vit_{scenario_name}.pth"

        if ckpt_path.exists():
            checkpoint = torch.load(ckpt_path, map_location=device, weights_only=True)
            model.load_state_dict(checkpoint)
        else:
            ax.set_title(f"{scenario_name}\n(No checkpoint)", fontsize=12)
            ax.axis('off')
            continue

        model.eval()

        # Collect predictions on test set
        all_true = []
        all_preds = []

        with torch.no_grad():
            for imgs, labels in test_loader:
                imgs = imgs.to(device)
                outputs = model(imgs)
                preds = outputs.argmax(dim=1).cpu().numpy()
                all_true.extend(labels.numpy())
                all_preds.extend(preds)

        all_true = np.array(all_true)
        all_preds = np.array(all_preds)

        # Compute confusion matrix with normalization (percentage)
        cm = confusion_matrix(all_true, all_preds, labels=list(range(num_classes)), normalize='true')

        # Plot heatmap
        im = ax.imshow(cm, cmap='Blues', vmin=0, vmax=1)
        ax.set_title(f"{scenario_name}\n(Test Acc: {(all_true == all_preds).mean()*100:.1f}%)",
                     fontsize=12, fontweight='bold')
        ax.set_xlabel('Predicted', fontsize=10)
        ax.set_ylabel('True', fontsize=10)

        # Add percentage annotations
        thresh = 0.5
        for i in range(num_classes):
            for j in range(num_classes):
                color = "white" if cm[i, j] > thresh else "black"
                ax.text(j, i, f"{cm[i, j]*100:.1f}%",
                        ha="center", va="center",
                        color=color, fontsize=9, fontweight='bold')

        # Set class labels
        ax.set_xticks(range(num_classes))
        ax.set_xticklabels(defect_classes, rotation=45, ha='right', fontsize=8)
        ax.set_yticks(range(num_classes))
        ax.set_yticklabels(defect_classes, fontsize=8)

        # Add colorbar
        cbar = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
        cbar.set_label('Percentage', rotation=270, labelpad=10)

    # Hide unused subplots
    for idx in range(num_scenarios, len(axes)):
        axes[idx].axis('off')

    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / "confusion_matrices_all_scenarios.png", dpi=150, bbox_inches='tight')
    plt.show()
    print(f"\nConfusion matrices saved to: {OUTPUT_DIR / 'confusion_matrices_all_scenarios.png'}")

In [ ]:
# --- Integrated Gradients Utilities ---
# Explain model decisions for defect classes using Integrated Gradients.

# IG configuration
IG_BASELINE_MODE = "zeros"
IG_VIZ_METHOD = "blended_heat_map"
IG_VIZ_SIGN = "all"
NON_DEFECT_CLASS_NAMES = {
    str(FULL_CLASSES[0]).strip().lower()
} if "FULL_CLASSES" in globals() and len(FULL_CLASSES) > 0 else {"good"}


# Convert image arrays to HWC format for visualization
def _to_hwc(arr: np.ndarray) -> np.ndarray:
    if arr.ndim == 4:
        if arr.shape[0] != 1:
            raise ValueError(f"Expected batch size 1, got shape={arr.shape}")
        arr = arr[0]

    if arr.ndim == 3 and arr.shape[0] <= 4 and arr.shape[1] > 4 and arr.shape[2] > 4:
        arr = np.transpose(arr, (1, 2, 0))
    elif arr.ndim == 2:
        arr = arr[..., np.newaxis]
    elif arr.ndim != 3:
        raise ValueError(f"Unsupported array shape: {arr.shape}")

    return arr


# Create baseline tensor for Integrated Gradients
def _ig_baseline(input_tensor: torch.Tensor) -> torch.Tensor:
    if IG_BASELINE_MODE == "zeros":
        return torch.zeros_like(input_tensor).to(device)
    raise ValueError(f"Unsupported IG_BASELINE_MODE: {IG_BASELINE_MODE}")


# Resolve target class for attribution
def _resolve_target_class(model: nn.Module, input_tensor: torch.Tensor, target_class: Optional[int] = None) -> int:
    if target_class is not None:
        return int(target_class)
    with torch.no_grad():
        output = model(input_tensor)
    return int(torch.argmax(output, dim=1).item())


# Compute IG attribution for one input image tensor
# Returns: (attributions, img_plot, target_id) in HWC numpy format
def apply_integrated_gradients(
    model: nn.Module,
    input_tensor: torch.Tensor,
    target_class: Optional[int] = None
) -> Tuple[np.ndarray, np.ndarray, int]:
    model.eval()
    ig_input = input_tensor.detach().clone().to(device).requires_grad_(True)
    ig = IntegratedGradients(model)
    baseline = _ig_baseline(ig_input)
    target_id = _resolve_target_class(model, ig_input, target_class)

    attributions, _ = ig.attribute(
        inputs=ig_input,
        baselines=baseline,
        target=target_id,
        return_convergence_delta=True,
    )

    attributions = _to_hwc(attributions.detach().cpu().numpy())
    img_plot = _to_hwc(ig_input.detach().cpu().numpy())
    return attributions, img_plot, target_id


# Prepare image for display with appropriate colormap
def _displayable_img(img_plot: np.ndarray) -> Tuple[np.ndarray, Optional[str]]:
    """Convert normalized tensor array to displayable image format."""
    # Denormalize from [-1, 1] back to [0, 1] if necessary
    if img_plot.min() < 0 or img_plot.max() > 1.0:
        img_plot = (img_plot + 1.0) / 2.0
        img_plot = np.clip(img_plot, 0.0, 1.0)

    if img_plot.ndim == 3 and img_plot.shape[-1] == 1:
        return img_plot[..., 0], "gray"
    if img_plot.ndim == 2:
        return img_plot, "gray"
    return img_plot, None


# Visualize IG attribution results with side-by-side heatmap and original image
def plot_ig_results(
    attributions: np.ndarray,
    img_plot: np.ndarray,
    predicted_class: int,
    actual_class: int,
    label_map: Dict[str, int],
    sample_idx: Optional[str] = None,
    save_path: Optional[Path] = None,
    model_name: str = "ViT"
) -> None:
    idx_to_class = {v: k for k, v in label_map.items()}
    pred_name = idx_to_class.get(int(predicted_class), str(predicted_class))
    actual_name = idx_to_class.get(int(actual_class), str(actual_class))
    sample_text = f"Sample {sample_idx}" if sample_idx is not None else "Sample"

    fig = plt.figure(figsize=(10, 5.6))
    fig.patch.set_facecolor("#ffffff")
    gs = fig.add_gridspec(2, 2, height_ratios=[20, 1], width_ratios=[1, 1], hspace=0.08, wspace=0.25)
    ax_heat = fig.add_subplot(gs[0, 0])
    ax_orig = fig.add_subplot(gs[0, 1])
    cax = fig.add_subplot(gs[1, :])

    viz.visualize_image_attr(
        attributions,
        img_plot,
        method=IG_VIZ_METHOD,
        sign=IG_VIZ_SIGN,
        show_colorbar=False,
        title="IG Heatmap",
        plt_fig_axis=(fig, ax_heat),
        use_pyplot=False,
    )
    ax_heat.set_title("IG Heatmap", color="#111111", fontsize=11)
    ax_heat.set_aspect("equal")
    if hasattr(ax_heat, "set_box_aspect"):
        ax_heat.set_box_aspect(1)

    disp_img, cmap = _displayable_img(img_plot)
    ax_orig.imshow(disp_img, cmap=cmap, interpolation="nearest")
    ax_orig.set_title("Original Image", color="#111111", fontsize=11)
    ax_orig.axis("off")
    ax_orig.set_aspect("equal")
    if hasattr(ax_orig, "set_box_aspect"):
        ax_orig.set_box_aspect(1)
    ax_orig.set_xlim(ax_heat.get_xlim())
    ax_orig.set_ylim(ax_heat.get_ylim())

    vmax = float(np.max(np.abs(attributions))) if attributions.size else 1.0
    if vmax == 0:
        vmax = 1.0
    ig_cmap = plt.get_cmap("RdYlGn")
    ig_norm = mcolors.Normalize(vmin=-vmax, vmax=vmax)
    sm = plt.cm.ScalarMappable(cmap=ig_cmap, norm=ig_norm)
    sm.set_array([])
    cbar = fig.colorbar(sm, cax=cax, orientation="horizontal")
    cbar.ax.tick_params(colors="#111111", labelsize=8)
    cbar.set_label("Red: negative | Green: positive", color="#111111", fontsize=9, labelpad=4)

    fig.suptitle(
        f"IG Attribution - {model_name} | Predicted: {pred_name} | Actual: {actual_name}",
        fontsize=12,
        color="#111111",
        y=0.98,
    )

    fig.subplots_adjust(top=0.86, bottom=0.18)
    if save_path is not None:
        save_path = Path(save_path)
        save_path.parent.mkdir(parents=True, exist_ok=True)
        fig.savefig(str(save_path), dpi=220, format="png", facecolor="#ffffff", edgecolor="#ffffff", bbox_inches="tight")
        print(f"Saved: {save_path}")
    plt.show()
    plt.close(fig)


# Extract defect class names (excluding non-defect classes like 'good')
def _defect_classes(label_map: Dict[str, int]) -> List[Tuple[str, int]]:
    classes = [
        (str(name), int(idx))
        for name, idx in label_map.items()
        if str(name).strip().lower() not in NON_DEFECT_CLASS_NAMES
    ]
    return classes if classes else [(str(n), int(i)) for n, i in label_map.items()]


# Create a safe filename token from text
def _safe_token(text: str) -> str:
    token = "".join(ch if (ch.isalnum() or ch in "-_") else "_" for ch in str(text))
    while "__" in token:
        token = token.replace("__", "_")
    return token.strip("_") or "na"


# Find first sample index for each target class efficiently
def _first_sample_index_per_class(dataset: Any, target_class_ids: Any) -> Dict[int, int]:
    sample_indices = {}

    if hasattr(dataset, "df") and hasattr(dataset, "label_map"):
        labels = dataset.df["indication_type"].astype(str).str.strip().tolist()
        for idx, class_name in enumerate(labels):
            label_id = int(dataset.label_map[class_name])
            if label_id in target_class_ids and label_id not in sample_indices:
                sample_indices[label_id] = idx
                if len(sample_indices) == len(target_class_ids):
                    break
        return sample_indices

    for idx in range(len(dataset)):  # type: ignore[arg-type]
        _, label = dataset[idx]
        label_id = int(label.item()) if torch.is_tensor(label) else int(label)
        if label_id in target_class_ids and label_id not in sample_indices:
            sample_indices[label_id] = idx
            if len(sample_indices) == len(target_class_ids):
                break
    return sample_indices


# Generate IG visualizations for one sample per defect class
def run_ig_demo_per_defect_class(
    model: nn.Module,
    dataset: Dataset,
    label_map: Dict[str, int],
    device: torch.device,
    output_dir: Path = OUTPUT_DIR,
    model_name: str = "ViT"
) -> None:
    print(f"\n{'='*60}")
    print(f"Running IG Attribution - {model_name}")
    print(f"{'='*60}")

    defect_classes = _defect_classes(label_map)
    target_class_ids = {class_id for _, class_id in defect_classes}
    sample_indices = _first_sample_index_per_class(dataset, target_class_ids)

    if not sample_indices:
        print("[WARN] No class samples found for IG visualization.")
        return

    print(f"Processing {len(sample_indices)} defect classes...\n")

    for class_name, class_id in defect_classes:
        if class_id not in sample_indices:
            continue

        ds_idx = sample_indices[class_id]
        sample_img, true_label = dataset[ds_idx]
        input_tensor = sample_img.unsqueeze(0).to(device)

        attrs, img_plot, pred_cls = apply_integrated_gradients(model, input_tensor)
        true_cls = int(true_label.item()) if torch.is_tensor(true_label) else int(true_label)

        save_file = output_dir / f"ig_{model_name.lower()}_{_safe_token(class_name)}_idx_{ds_idx}.png"
        plot_ig_results(
            attrs, img_plot, pred_cls, true_cls, label_map,
            sample_idx=f"{class_name} (idx={ds_idx})",
            save_path=save_file,
            model_name=model_name,
        )

    print(f"\n{'='*60}")
    print(f"Complete. Results saved to: {output_dir}")
    print(f"{'='*60}")


In [ ]:
# --- IG Run for ViT ---
# Captum import is already at top of notebook; verify availability here
try:
    _ = IntegratedGradients  # Will raise NameError if not imported
    _captum_ready = True
except NameError:
    try:
        from captum.attr import IntegratedGradients
        from captum.attr import visualization as viz
        _captum_ready = True
    except ImportError:
        IntegratedGradients = None
        viz = None
        _captum_ready = False

if not _captum_ready:
    print("[INFO] Captum not available. Install with: pip install captum")
else:
    # Detect ViT model from global namespace
    model_candidates = [
        ("model", globals().get("model")),
        ("vit_model", globals().get("vit_model")),
        ("model_vit", globals().get("model_vit")),
    ]

    selected_model = None
    model_var_name = None

    for var_name, candidate in model_candidates:
        if candidate is not None and isinstance(candidate, nn.Module):
            # Check if it's a ViT model by class name or variable name
            if "ViT" in type(candidate).__name__ or "vit" in var_name.lower():
                selected_model = candidate
                model_var_name = var_name
                break

    # Fallback to any available model
    if selected_model is None and model_candidates[0][1] is not None:
        selected_model = model_candidates[0][1]
        model_var_name = "model"

    train_ds = globals().get("train_ds")
    label_map = globals().get("label_map")

    # Check if all prerequisites are ready
    if selected_model is not None and label_map is not None and train_ds is not None and len(train_ds) > 0:  # type: ignore[arg-type]
        print(f"[OK] Using model: '{model_var_name}' ({type(selected_model).__name__})")
        try:
            # Type assertions to satisfy type checker
            assert isinstance(label_map, dict), "label_map must be a dict"
            assert train_ds is not None, "train_ds must not be None"
            run_ig_demo_per_defect_class(
                selected_model,
                train_ds,  # type: ignore[arg-type]
                label_map,
                device,
                model_name="ViT"
            )
        except Exception as e:
            print(f"[ERROR] IG demo failed: {e}")
            import traceback
            traceback.print_exc()
    else:
        print("[INFO] Prerequisites not ready. Run ViT training cell first.")


In [ ]:
# --- IG Baseline Comparison: Trained vs Untrained ViT ---
# Generate IG attribution maps for both trained and random (untrained) ViT models
# to distinguish learned attributions from random image-based artifacts

try:
    _ = IntegratedGradients  # Will raise NameError if not imported
    _captum_ready = True
except NameError:
    try:
        from captum.attr import IntegratedGradients
        from captum.attr import visualization as viz
        _captum_ready = True
    except ImportError:
        IntegratedGradients = None
        viz = None
        _captum_ready = False

if not _captum_ready:
    print("[INFO] Captum not available. Install with: pip install captum")
else:
    # Use the best trained ViT model (from scenario with highest F1)
    if not scenario_results:
        print("[ERROR] No trained scenarios found. Run training first.")
    else:
        # Find best scenario
        best_scenario_name = max(scenario_results, key=lambda x: scenario_results[x]["best_metrics"]["f1"])
        best_scenario = scenario_results[best_scenario_name]
        best_ckpt_path = Path(best_scenario["checkpoint"])
        
        print(f"\n{'='*80}")
        print(f"IG BASELINE COMPARISON: Trained vs Untrained ViT")
        print(f"{'='*80}")
        print(f"Using best trained model: {best_scenario_name} (F1: {best_scenario['best_metrics']['f1']:.4f})")
        print(f"Checkpoint: {best_ckpt_path}")
        
        # Create output directory for baseline comparison
        ig_baseline_dir = OUTPUT_DIR / "ig_baseline_comparison"
        ig_baseline_dir.mkdir(parents=True, exist_ok=True)
        
        # Load the trained model
        label_map = load_or_create_label_map(OUT_CSV)
        trained_model = create_vit_model(num_classes=len(label_map)).to(device)
        trained_model.load_state_dict(torch.load(best_ckpt_path, map_location=device, weights_only=True))
        trained_model.eval()
        print(f"\nLoaded trained model from: {best_ckpt_path}")
        
        # Create untrained (random) model with same architecture
        torch.manual_seed(42)  # For reproducibility of random weights
        untrained_model = create_vit_model(num_classes=len(label_map)).to(device)
        # Re-initialize with random weights (no pretraining)
        from torchvision.models import vit_b_16
        untrained_model = vit_b_16(weights=None)
        untrained_model.heads.head = nn.Linear(untrained_model.heads.head.in_features, len(label_map))
        untrained_model = untrained_model.to(device)
        untrained_model.eval()
        print("Created untrained model with random weights")
        
        # Use validation or training dataset for comparison
        dataset = globals().get("vit_val_ds") or globals().get("vit_train_ds") or globals().get("train_ds")
        if dataset is None:
            print("[ERROR] No dataset available")
        else:
            print(f"Using dataset: {type(dataset).__name__} (size: {len(dataset)})")
            
            # Get defect classes
            defect_classes_list = [
                (str(name), int(idx))
                for name, idx in label_map.items()
                if str(name).strip().lower() not in {"good"}
            ]
            
            if not defect_classes_list:
                print("[WARN] No defect classes found in label map")
            else:
                print(f"\nProcessing {len(defect_classes_list)} defect classes...")
                print(f"Generating IG maps for both trained and untrained models...\n")
                
                # Find first sample for each defect class
                target_class_ids = {class_id for _, class_id in defect_classes_list}
                sample_indices = _first_sample_index_per_class(dataset, target_class_ids)
                
                if not sample_indices:
                    print("[WARN] No samples found for defect classes")
                else:
                    # Process each defect class
                    for class_name, class_id in defect_classes_list:
                        if class_id not in sample_indices:
                            continue
                        
                        ds_idx = sample_indices[class_id]
                        sample_img, true_label = dataset[ds_idx]
                        input_tensor = sample_img.unsqueeze(0).to(device)
                        
                        # Get predictions
                        with torch.no_grad():
                            trained_output = trained_model(input_tensor)
                            trained_pred = int(torch.argmax(trained_output, dim=1).item())
                            
                            untrained_output = untrained_model(input_tensor)
                            untrained_pred = int(torch.argmax(untrained_output, dim=1).item())
                        
                        true_cls = int(true_label.item()) if torch.is_tensor(true_label) else int(true_label)
                        
                        # Compute IG for trained model
                        trained_attrs, trained_img, _ = apply_integrated_gradients(
                            trained_model, input_tensor, target_class=class_id
                        )
                        
                        # Compute IG for untrained model
                        untrained_attrs, untrained_img, _ = apply_integrated_gradients(
                            untrained_model, input_tensor, target_class=class_id
                        )
                        
                        # Create side-by-side comparison visualization
                        idx_to_class = {v: k for k, v in label_map.items()}
                        actual_name = idx_to_class.get(true_cls, str(true_cls))
                        trained_pred_name = idx_to_class.get(trained_pred, str(trained_pred))
                        untrained_pred_name = idx_to_class.get(untrained_pred, str(untrained_pred))
                        
                        # Create 2x2 figure: [Trained IG] [Original] [Untrained IG] [Original]
                        fig = plt.figure(figsize=(16, 7))
                        fig.patch.set_facecolor("#ffffff")
                        gs = fig.add_gridspec(2, 4, height_ratios=[20, 1], wspace=0.15, hspace=0.08)
                        
                        # Trained IG heatmap
                        ax_trained_heat = fig.add_subplot(gs[0, 0])
                        viz.visualize_image_attr(
                            trained_attrs,
                            trained_img,
                            method=IG_VIZ_METHOD,
                            sign=IG_VIZ_SIGN,
                            show_colorbar=False,
                            title="",
                            plt_fig_axis=(fig, ax_trained_heat),
                            use_pyplot=False,
                        )
                        ax_trained_heat.set_title("Trained ViT IG", color="#111111", fontsize=11, fontweight='bold')
                        ax_trained_heat.set_aspect("equal")
                        
                        # Original image (left)
                        ax_orig1 = fig.add_subplot(gs[0, 1])
                        disp_img1, cmap1 = _displayable_img(trained_img)
                        ax_orig1.imshow(disp_img1, cmap=cmap1, interpolation="nearest")
                        ax_orig1.set_title("Original Image", color="#111111", fontsize=11)
                        ax_orig1.axis("off")
                        ax_orig1.set_aspect("equal")
                        
                        # Untrained IG heatmap
                        ax_untrained_heat = fig.add_subplot(gs[0, 2])
                        viz.visualize_image_attr(
                            untrained_attrs,
                            untrained_img,
                            method=IG_VIZ_METHOD,
                            sign=IG_VIZ_SIGN,
                            show_colorbar=False,
                            title="",
                            plt_fig_axis=(fig, ax_untrained_heat),
                            use_pyplot=False,
                        )
                        ax_untrained_heat.set_title("Untrained ViT IG (Random)", color="#111111", fontsize=11, fontweight='bold')
                        ax_untrained_heat.set_aspect("equal")
                        
                        # Original image (right)
                        ax_orig2 = fig.add_subplot(gs[0, 3])
                        disp_img2, cmap2 = _displayable_img(untrained_img)
                        ax_orig2.imshow(disp_img2, cmap=cmap2, interpolation="nearest")
                        ax_orig2.set_title("Original Image", color="#111111", fontsize=11)
                        ax_orig2.axis("off")
                        ax_orig2.set_aspect("equal")
                        
                        # Shared colorbar
                        cax = fig.add_subplot(gs[1, :])
                        vmax_trained = float(np.max(np.abs(trained_attrs))) if trained_attrs.size else 1.0
                        vmax_untrained = float(np.max(np.abs(untrained_attrs))) if untrained_attrs.size else 1.0
                        vmax = max(vmax_trained, vmax_untrained)
                        if vmax == 0:
                            vmax = 1.0
                        ig_cmap = plt.get_cmap("RdYlGn")
                        ig_norm = mcolors.Normalize(vmin=-vmax, vmax=vmax)
                        sm = plt.cm.ScalarMappable(cmap=ig_cmap, norm=ig_norm)
                        sm.set_array([])
                        cbar = fig.colorbar(sm, cax=cax, orientation="horizontal")
                        cbar.ax.tick_params(colors="#111111", labelsize=8)
                        cbar.set_label("Red: negative | Green: positive", color="#111111", fontsize=9, labelpad=4)
                        
                        # Title with class info and predictions
                        fig.suptitle(
                            f"IG Baseline Comparison - Class: {class_name} | Actual: {actual_name} | "
                            f"Trained Pred: {trained_pred_name} | Untrained Pred: {untrained_pred_name}",
                            fontsize=12,
                            color="#111111",
                            y=0.98,
                            fontweight='bold'
                        )
                        
                        fig.subplots_adjust(top=0.86, bottom=0.18)
                        
                        # Save figure
                        save_file = ig_baseline_dir / f"ig_baseline_{_safe_token(class_name)}_idx_{ds_idx}.png"
                        fig.savefig(str(save_file), dpi=220, format="png", facecolor="#ffffff", edgecolor="#ffffff", bbox_inches="tight")
                        print(f"Saved: {save_file}")
                        
                        plt.show()
                        plt.close(fig)
                    
                    print(f"\n{'='*80}")
                    print(f"BASELINE COMPARISON COMPLETE")
                    print(f"{'='*80}")
                    print(f"Results saved to: {ig_baseline_dir}")
                    print(f"\nComparison shows:")
                    print(f"  - Left: IG from trained ViT (learned features)")
                    print(f"  - Right: IG from untrained ViT (random weights, image artifacts)")
                    print(f"{'='*80}\n")

